In [31]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import re

In [ ]:
drug_mapping = pd.read_csv('../../../data/vocab/drug-mappings.tsv', sep='\t')
drug_mapping.head()

In [33]:
def get_kegg_pathway(text: str) -> list[str]:
    ids = []
    start = False

    for line in text.splitlines():
        if start and re.match(r'^\s*[A-Z]{2,}\b', line):
            break

        if re.match(r'^\s*PATHWAY\b', line):
            start = True
            # e.g. "    PATHWAY     hsa04080(1137+1141)  Neuroactive..."
            for m in re.finditer(r'(\w+\d+)\(', line):
                ids.append(m.group(1))
            continue

        if start:
            m = re.match(r'^\s*(\w+\d+)\(', line)
            if m:
                ids.append(m.group(1))

    return ids

In [ ]:
data = []

import time

def get(kegg_id):
    r = requests.get(f'https://rest.kegg.jp/get/{kegg_id}')
    if r.status_code == 200:
        return {
            'kegg_id': kegg_id,
            'drugbank_id': drugbank_id,
            'data': r.text
        }
    else:
        print(f'retry {kegg_id}')
        time.sleep(60)
        return get(kegg_id)

for idx, row in drug_mapping.iterrows():
    print(f'{idx}/{len(drug_mapping)}', end='\r')
    drugbank_id = row['drugbankId']
    kegg_id = row['kegg_id']
    if kegg_id is not np.nan:
        data.append(get(kegg_id))

df = pd.DataFrame(data)
df = df.dropna()
df = df.drop_duplicates()
df.to_csv('../../data/kegg/kegg_drug.csv', index=False)

In [ ]:
patway_mapping = pd.read_csv('../../../data/vocab/kegg_reactome.csv')
patway_mapping.head()

In [ ]:
patway_mapping['Mapping Type'].unique()

In [ ]:
df = pd.read_csv('../../../data/kegg/kegg_drug.csv', low_memory=False)

data = []

def get_name(pathway):
    r = requests.get(f'https://rest.kegg.jp/get/{pathway}')
    if r.status_code == 200:
        return r.text.split('\n')[1].lstrip('NAME').strip()
    else:
        print(f'retry {pathway}')
        time.sleep(60)
        return get(pathway)

name_map = {}

for idx, row in df.iterrows():
    print(f'{idx}/{len(df)}', end='\r')
    drugbank_id = row['drugbank_id']
    kegg_id = row['kegg_id']
    text = row['data']
    pathways = get_kegg_pathway(text)
    pathways_reactome = []
    for pathway in pathways:
        if pathway not in name_map:
            name_map[pathway] = get_name(pathway)
        name = name_map[pathway]
        if '- Homo sapiens (human)' not in name:
            continue
        data.append({
            'drugbank_id': drugbank_id,
            'pathway_kegg_id': pathway,
            'pathway_kegg_name': name.rstrip('- Homo sapiens (human)')
        })

df = pd.DataFrame(data)
df = df.dropna()
df = df.drop_duplicates()
df.to_csv('../../../data/kegg/kegg_drug_pathway.csv', index=False)

In [39]:
df_kegg_drug_pathway = pd.read_csv('../../../data/kegg/kegg_drug_pathway.csv', low_memory=False)

drug_kegg_pathway = []
drug_reactome_pathway = []
reactome_kegg = []

for idx, row in df_kegg_drug_pathway.iterrows():
    print(f'{idx}/{len(df_kegg_drug_pathway)}', end='\r')
    pathway_kegg_id = row['pathway_kegg_id']
    drugbank_id = row['drugbank_id']
    pathway_kegg_name = row['pathway_kegg_name']
    
    kegg_source = f'(`Source Resource` == "kegg.pathway" and `Source ID` == "path:{pathway_kegg_id}")'
    kegg_target = f'(`Target Resource` == "kegg.pathway" and `Target ID` == "path:{pathway_kegg_id}")'
    mapping_type_part = f'(`Mapping Type` == "isPartOf")'
    mapping_type_equivalent = f'(`Mapping Type` == "equivalentTo")'
    
    if not (res1 := patway_mapping.query(f'{kegg_source} and {mapping_type_equivalent}')).empty:
        drug_reactome_pathway.append({
            'drugbank_id': drugbank_id,
            'pathway_reactome_id': res1.iloc[0]['Target ID'],
            'pathway_reactome_name': res1.iloc[0]['Target Name']
        })
    elif not (res2 := patway_mapping.query(f'{kegg_target} and {mapping_type_equivalent}')).empty:
        drug_reactome_pathway.append({
            'drugbank_id': drugbank_id,
            'pathway_reactome_id': res2.iloc[0]['Source ID'],
            'pathway_reactome_name': res2.iloc[0]['Source Name']
        })
    elif not (res3 := patway_mapping.query(f'{kegg_source} and {mapping_type_part}')).empty:
        drug_kegg_pathway.append({
            'drugbank_id': drugbank_id,
            'pathway_kegg_id': pathway_kegg_id,
            'pathway_kegg_name': pathway_kegg_name
        })
        reactome_kegg.append({
            'relation': 'pathway_pathway',
            'display_relation': 'parent-child',
            'x_id': res3.iloc[0]['Target ID'],
            'x_name': res3.iloc[0]['Target Name'],
            'x_source': 'REACTOME',
            'x_type': 'pathway',
            'y_id': pathway_kegg_id,
            'y_name': pathway_kegg_name,
            'y_source': 'KEGG',
            'y_type': 'pathway'
        })
    elif not (res4 := patway_mapping.query(f'{kegg_target} and {mapping_type_part}')).empty:
        drug_kegg_pathway.append({
            'drugbank_id': drugbank_id,
            'pathway_kegg_id': pathway_kegg_id,
            'pathway_kegg_name': pathway_kegg_name
        })
        reactome_kegg.append({
            'relation': 'pathway_pathway',
            'display_relation': 'parent-child',
            'x_id': pathway_kegg_id,
            'x_name': pathway_kegg_name,
            'x_source': 'KEGG',
            'x_type': 'pathway',
            'y_id': res4.iloc[0]['Source ID'],
            'y_name': res4.iloc[0]['Source Name'],
            'y_source': 'REACTOME',
            'y_type': 'pathway'
        }) 

pd.DataFrame(drug_kegg_pathway).to_csv('../../../data/kegg/drug_kegg_pathway.csv', index=False)
pd.DataFrame(reactome_kegg).to_csv('../../../data/kegg/reactome_kegg.csv', index=False)
pd.DataFrame(drug_reactome_pathway).to_csv('../../../data/kegg/drug_reactome_pathway.csv', index=False)